# SimpleShot EXACT, Run repo's original code with pickle data


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
WORK_DIR = '/content/drive/MyDrive/CS5782/SimpleShot/SImpleShot Reproduce_WOODY_CHANG'
DATA_DIR = '/content/drive/MyDrive/SimpleShot'  # where pkl files are
SAVE_DIR = os.path.join(WORK_DIR, 'results_exact')

# Clone repo if needed
repo_dir = os.path.join(WORK_DIR, 'simple_shot')
if not os.path.exists(repo_dir):
    os.chdir(WORK_DIR)
    !git clone https://github.com/mileyan/simple_shot.git

os.chdir(repo_dir)
print('Working dir:', os.getcwd())
!pip install configargparse tqdm -q

Mounted at /content/drive
Working dir: /content/drive/MyDrive/CS5782/SimpleShot/SImpleShot Reproduce_WOODY_CHANG/simple_shot


In [ ]:
# Replace ONLY loader.py to read from pickle
loader_code = '''import os
import pickle
import numpy as np
import PIL.Image as Image

__all__ = ["DatasetFolder"]

class DatasetFolder(object):
    def __init__(self, root, split_dir, split_type, transform, out_name=False):
        pkl_map = {"train": "mini-imagenet-cache-train.pkl",
                   "val": "mini-imagenet-cache-val.pkl",
                   "test": "mini-imagenet-cache-test.pkl"}
        pkl_path = os.path.join(root, pkl_map[split_type])
        with open(pkl_path, "rb") as f:
            pkl_data = pickle.load(f)
        images = pkl_data["image_data"]
        class_dict = pkl_data["class_dict"]
        label_key = sorted(class_dict.keys())
        label_map = {cls: i for i, cls in enumerate(label_key)}
        self.images = []
        self.labels = []
        self.data = []
        for cls_name in label_key:
            for idx in sorted(class_dict[cls_name]):
                self.images.append(images[idx])
                self.labels.append(label_map[cls_name])
                self.data.append(f"{cls_name}_{idx}.jpg")
        self.transform = transform
        self.out_name = out_name
        self.length = len(self.images)

    def __len__(self):
        return self.length

    def __getitem__(self, index):
        img = Image.fromarray(self.images[index])
        label = self.labels[index]
        if self.transform:
            img = self.transform(img)
        if self.out_name:
            return img, label, self.data[index]
        return img, label
'''

with open('src/datasets/loader.py', 'w') as f:
    f.write(loader_code)
print('Replaced loader.py')

Replaced loader.py


In [ ]:
# Fix PyTorch compatibility: .view() -> .reshape()
train_py = 'src/train.py'
with open(train_py, 'r') as f:
    code = f.read()
code = code.replace(
    'correct_k = correct[:k].view(-1).float().sum(0, keepdim=True)',
    'correct_k = correct[:k].reshape(-1).float().sum(0, keepdim=True)'
)
with open(train_py, 'w') as f:
    f.write(code)
print('Fixed .view() -> .reshape()')

Fixed .view() -> .reshape()


In [ ]:
# Train using repo's code
!python src/train.py \
    -c configs/mini/softmax/densenet121.config \
    --data "{DATA_DIR}/" \
    --workers 2 \
    --save-path "{SAVE_DIR}" \
    --disable-tqdm

/content/drive/MyDrive/CS5782/SimpleShot/SImpleShot Reproduce_WOODY_CHANG/simple_shot/src/train.py:309: SyntaxWarning: "is not" with 'str' literal. Did you mean "!="?
  if os.path.dirname(filepath) is not '':
Meta Test: LAST
feature	UN	L2N	CL2N
GVP 1Shot	0.5317(0.0020)	0.5683(0.0020)	0.5963(0.0020)
GVP_5Shot	0.7571(0.0015)	0.7636(0.0015)	0.7679(0.0015)
Meta Test: BEST
feature	UN	L2N	CL2N
GVP 1Shot	0.5362(0.0020)	0.5732(0.0020)	0.5972(0.0020)
GVP_5Shot	0.7610(0.0015)	0.7680(0.0015)	0.7714(0.0015)


In [ ]:
!tail -20 "{SAVE_DIR}/training.log"

[2026-04-12 01:38:54 train.py:38] INFO     resume: 
[2026-04-12 01:38:54 train.py:38] INFO     save_path: /content/drive/MyDrive/CS5782/SimpleShot/SImpleShot Reproduce_WOODY_CHANG/results_exact
[2026-04-12 01:38:54 train.py:38] INFO     scheduler: multi_step
[2026-04-12 01:38:54 train.py:38] INFO     seed: None
[2026-04-12 01:38:54 train.py:38] INFO     split_dir: ./split/mini/
[2026-04-12 01:38:54 train.py:38] INFO     start_epoch: 0
[2026-04-12 01:38:54 train.py:38] INFO     weight_decay: 0.0001
[2026-04-12 01:38:54 train.py:38] INFO     workers: 2
[2026-04-12 01:38:54 train.py:46] INFO     => creating model 'densenet121'
[2026-04-12 01:38:54 train.py:49] INFO     Number of model parameters: 7011648
[2026-04-12 01:38:54 train.py:77] INFO     => loading checkpoint '/content/drive/MyDrive/CS5782/SimpleShot/SImpleShot Reproduce_WOODY_CHANG/results_exact/checkpoint.pth.tar'
[2026-04-12 01:38:54 train.py:84] INFO     => loaded checkpoint '/content/drive/MyDrive/CS5782/SimpleShot/SImpleSho

In [ ]:
# Evaluate with --enlarge
!python src/train.py \
    -c configs/mini/softmax/densenet121.config \
    --data "{DATA_DIR}/" \
    --workers 2 \
    --save-path "{SAVE_DIR}" \
    --batch-size 64 \
    --evaluate \
    --enlarge \
    --disable-tqdm

/content/drive/MyDrive/CS5782/SimpleShot/SImpleShot Reproduce_WOODY_CHANG/simple_shot/src/train.py:309: SyntaxWarning: "is not" with 'str' literal. Did you mean "!="?
  if os.path.dirname(filepath) is not '':
Meta Test: LAST
feature	UN	L2N	CL2N
GVP 1Shot	0.5170(0.0021)	0.5485(0.0020)	0.5670(0.0020)
GVP_5Shot	0.7250(0.0016)	0.7339(0.0016)	0.7356(0.0016)
Meta Test: BEST
feature	UN	L2N	CL2N
GVP 1Shot	0.5451(0.0020)	0.5828(0.0020)	0.6074(0.0020)
GVP_5Shot	0.7685(0.0015)	0.7770(0.0015)	0.7800(0.0015)
